# Propositional interpretable baseline: decision tree

An additional experiment for the KG-Microbe case study, requested during review.

The manuscript compares RDFRules against a baseline that differs from it in two ways at once: it is propositional rather than relational, and a black-box ensemble rather than a set of readable rules. The observed gap can therefore not be attributed to either difference. This notebook supplies the missing combination, a model that is propositional **and** interpretable, so that the two effects can be told apart.

A single decision tree is trained on the tabular data already committed to this repository and scored on the same test cohort with the same metrics. Its depth is limited to 4 to match the RDFRules maximum rule length of 4 (3 body atoms + 1 head), so that one root-to-leaf path costs the reader roughly what one mined rule costs.

## Setup

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, f1_score, classification_report

DATA = "../data"
RANDOM_SEED = 12
MAX_DEPTH = 4

## Target variable

The `y_*.tsv` label files exported by `Catboost_reference_notebook.ipynb` are not part of the repository, so the medium of each organism is rebuilt from the graph using the same steps as that notebook: keep `NCBITaxon` &rarr; `medium` edges, drop media whose name mentions agar, drop the explicitly excluded media, and keep only organisms left with exactly one medium.

In [2]:
EXCLUDED_MEDIA = [
    "medium:104", "medium:1a", "medium:215", "medium:381", "medium:545",
    "medium:553", "medium:830c", "medium:J12", "medium:514c",
]

nodes = pd.read_csv(f"{DATA}/merged-kg_nodes.tsv", sep="\t", usecols=["id", "name"], dtype=str)
edges = pd.read_csv(f"{DATA}/merged-kg_edges.tsv", sep="\t", usecols=["subject", "object"], dtype=str)

medium_names = nodes.set_index("id")["name"]
agar_media = set(medium_names[medium_names.str.contains("agar", case=False, na=False)].index)

organism_medium_edges = edges[
    edges["subject"].str.startswith("NCBITaxon:", na=False)
    & edges["object"].str.startswith("medium:", na=False)
].drop_duplicates()

organism_medium_edges = organism_medium_edges[
    ~organism_medium_edges["object"].isin(agar_media)
    & ~organism_medium_edges["object"].isin(EXCLUDED_MEDIA)
]

media_per_organism = organism_medium_edges.groupby("subject")["object"].agg(set)
targets = media_per_organism[media_per_organism.apply(len) == 1].apply(min)

print(f"organisms with exactly one medium: {len(targets)}")

organisms with exactly one medium: 7743


The committed `catboost_*.tsv` splits are used unchanged, so the tree sees exactly the instances and features the tabular baseline saw. The test cohort is the 407 cases reported in the manuscript; the committed training and validation files hold 2,872 and 824 instances, and ten media pass the minimum class support of 40.

In [3]:
def load_split(split):
    features = pd.read_csv(f"{DATA}/catboost_{split}.tsv", sep="\t", index_col="subject")
    return features, targets.reindex(features.index)


X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

majority_baseline = y_test.value_counts().iloc[0] / len(y_test)

print(f"train {X_train.shape}, validation {X_val.shape}, test {X_test.shape}")
print(f"classes: {y_train.nunique()}, majority class on test: {majority_baseline:.2%}")

train (2872, 568), validation (824, 568), test (407, 568)
classes: 10, majority class on test: 28.99%


## Training and evaluation

Scored on the same test cohort and with the same metrics reported in the manuscript: Hits@1 (accuracy) overall, precision, recall and F1 per medium.

In [4]:
tree = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=RANDOM_SEED).fit(X_train, y_train)
y_pred = tree.predict(X_test)

print(f"Hits@1: {accuracy_score(y_test, y_pred):.2%}")
print(f"macro F1: {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f}")

Hits@1: 48.89%
macro F1: 0.265


### Per-class scores

Precision, recall and F1 per medium, in the format of Table 4 of the manuscript.

In [5]:
report = pd.DataFrame(
    classification_report(y_test, y_pred, output_dict=True, zero_division=0)
).T.loc[y_test.unique()]

report.insert(0, "medium_name", medium_names.reindex(report.index))
report.sort_values("support", ascending=False).round(3)

,medium_name,precision,recall,f1-score,support
medium:514,BACTO MARINE BROTH (DIFCO 2216) (DSMZ Medium 514),0.461,0.847,0.597,118.0
medium:830,R2A MEDIUM (DSMZ Medium 830),0.653,0.395,0.492,81.0
medium:65,GYM STREPTOMYCES MEDIUM (DSMZ Medium 65),0.481,0.591,0.531,66.0
medium:92,TRYPTICASE SOY YEAST EXTRACT MEDIUM (DSMZ Medi...,0.368,0.135,0.197,52.0
medium:693,COLUMBIA BLOOD MEDIUM (DSMZ Medium 693),0.457,0.381,0.416,42.0
medium:11,MRS MEDIUM (DSMZ Medium 11),0.833,0.278,0.417,18.0
medium:645,MIDDLEBROOK MEDIUM (DSMZ Medium 645),0.000,0.000,0.000,11.0
medium:J14,EG MEDIUM,0.000,0.000,0.000,8.0
medium:1076b,SP4-Z MEDIUM (DSMZ Medium 1076b),0.000,0.000,0.000,6.0
medium:98,RHIZOBIUM MEDIUM (DSMZ Medium 98),0.000,0.000,0.000,5.0


### Relaxing the depth constraint

The experiment specification allows a depth of 3 or 4; both are reported below, with depth 8 for reference.

In [6]:
for depth in [3, 4, 8]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED).fit(X_train, y_train)
    validation = accuracy_score(y_val, model.predict(X_val))
    test = accuracy_score(y_test, model.predict(X_test))
    print(f"max_depth={str(depth):5} validation {validation:.2%}, test {test:.2%}")

max_depth=3     validation 44.42%, test 46.19%
max_depth=4     validation 51.09%, test 48.89%
max_depth=8     validation 57.28%, test 58.97%


### Decision paths

The IF-THEN splits the tree uses, for the side-by-side comparison against a multi-hop relational rule.

In [7]:
print(export_text(tree, feature_names=list(X_train.columns)))

|--- gram_stain:negative <= 0.50
|   |--- oxygen:microaerophile <= 0.50
|   |   |--- temp_opt:mid2 <= 0.50
|   |   |   |--- gc:high <= 0.50
|   |   |   |   |--- class: medium:514
|   |   |   |--- gc:high >  0.50
|   |   |   |   |--- class: medium:65
|   |   |--- temp_opt:mid2 >  0.50
|   |   |   |--- cell_shape:rod <= 0.50
|   |   |   |   |--- class: medium:65
|   |   |   |--- cell_shape:rod >  0.50
|   |   |   |   |--- class: medium:92
|   |--- oxygen:microaerophile >  0.50
|   |   |--- CHEBI:59640 <= 0.50
|   |   |   |--- EC:1.11.1.6 <= 0.50
|   |   |   |   |--- class: medium:693
|   |   |   |--- EC:1.11.1.6 >  0.50
|   |   |   |   |--- class: medium:693
|   |   |--- CHEBI:59640 >  0.50
|   |   |   |--- EC:3.1.3.1 <= 0.50
|   |   |   |   |--- class: medium:11
|   |   |   |--- EC:3.1.3.1 >  0.50
|   |   |   |   |--- class: medium:693
|--- gram_stain:negative >  0.50
|   |--- NaCl_range:mid <= 0.50
|   |   |--- NaCl_range:very_low <= 0.50
|   |   |   |--- oxygen:microaerophile <= 0.50


## Results

| Model | Representation | Interpretable | Hits@1 |
| --- | --- | --- | --- |
| Majority class | &mdash; | &mdash; | 28.99% |
| Decision tree (depth 4) | propositional, single-hop | yes | 48.89% |
| RDFRules | relational, multi-hop | yes | 64.37% |
| CatBoost | propositional, single-hop | no | 72.00% |

